# 📉 Telco Customer Churn Prediction

## 🧠 Business Problem

Customer churn is one of the biggest challenges for telecom companies. Acquiring a new customer is significantly more expensive than retaining an existing one. Therefore, identifying customers who are likely to churn allows companies to take proactive actions and reduce revenue loss.

---

## 🎯 Objective

The objective of this project is to build a machine learning model that can predict whether a customer is likely to churn based on historical customer data.

---

## 💼 Business Impact

This model can help the business:

- Identify high-risk customers before they leave  
- Enable targeted retention campaigns  
- Optimize marketing spend by focusing on the right customers  
- Improve overall customer lifetime value (CLTV)

---

## 📌 Problem Type

- Binary Classification Problem  
- Target Variable: **Churn (Yes / No)**

---

## 📚 Step 1: Import Required Libraries

In this section, we import all the necessary libraries for data manipulation, visualization, and model building.

In [1]:
# Core libraries for data manipulation and visualization

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

## 📥 Step 2: Load Dataset

In this step, we load the Telco Customer Churn dataset and perform an initial inspection to understand its structure.

We will examine:
- Number of rows and columns  
- Sample records  
- Basic structure of the dataset  

In [2]:
# Load dataset
df = pd.read_excel("../dataset/Telco_customer_churn.xlsx")

# Preview first few rows
df.head()

,CustomerID,Count,Country,State,City,Zip Code,Lat Long,Latitude,Longitude,Gender,...,Contract,Paperless Billing,Payment Method,Monthly Charges,Total Charges,Churn Label,Churn Value,Churn Score,CLTV,Churn Reason
0,3668-QPYBK,1,United States,California,Los Angeles,90003,"33.964131, -118.272783",33.964131,-118.272783,Male,...,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,1,86,3239,Competitor made better offer
1,9237-HQITU,1,United States,California,Los Angeles,90005,"34.059281, -118.30742",34.059281,-118.307420,Female,...,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,1,67,2701,Moved
2,9305-CDSKC,1,United States,California,Los Angeles,90006,"34.048013, -118.293953",34.048013,-118.293953,Female,...,Month-to-month,Yes,Electronic check,99.65,820.5,Yes,1,86,5372,Moved
3,7892-POOKP,1,United States,California,Los Angeles,90010,"34.062125, -118.315709",34.062125,-118.315709,Female,...,Month-to-month,Yes,Electronic check,104.80,3046.05,Yes,1,84,5003,Moved
4,0280-XJGEX,1,United States,California,Los Angeles,90015,"34.039224, -118.266293",34.039224,-118.266293,Male,...,Month-to-month,Yes,Bank transfer (automatic),103.70,5036.3,Yes,1,89,5340,Competitor had better devices


In [4]:
# Check dataset shape
df.shape

(7043, 33)

### 📊 Observation

- Dataset contains **7043 rows and 33 columns**
- Each row represents a unique customer

## 🔍 Step 3: Data Understanding

In this section, we analyze the dataset to understand:

- Feature types  
- Unique values  
- Redundant columns  
- Potential data issues  

This helps in deciding what to keep, drop, or transform.

In [5]:
pd.set_option('display.max_columns', None)
df.head()

,CustomerID,Count,Country,State,City,Zip Code,Lat Long,Latitude,Longitude,Gender,Senior Citizen,Partner,Dependents,Tenure Months,Phone Service,Multiple Lines,Internet Service,Online Security,Online Backup,Device Protection,Tech Support,Streaming TV,Streaming Movies,Contract,Paperless Billing,Payment Method,Monthly Charges,Total Charges,Churn Label,Churn Value,Churn Score,CLTV,Churn Reason
0,3668-QPYBK,1,United States,California,Los Angeles,90003,"33.964131, -118.272783",33.964131,-118.272783,Male,No,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,1,86,3239,Competitor made better offer
1,9237-HQITU,1,United States,California,Los Angeles,90005,"34.059281, -118.30742",34.059281,-118.307420,Female,No,No,Yes,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,1,67,2701,Moved
2,9305-CDSKC,1,United States,California,Los Angeles,90006,"34.048013, -118.293953",34.048013,-118.293953,Female,No,No,Yes,8,Yes,Yes,Fiber optic,No,No,Yes,No,Yes,Yes,Month-to-month,Yes,Electronic check,99.65,820.5,Yes,1,86,5372,Moved
3,7892-POOKP,1,United States,California,Los Angeles,90010,"34.062125, -118.315709",34.062125,-118.315709,Female,No,Yes,Yes,28,Yes,Yes,Fiber optic,No,No,Yes,Yes,Yes,Yes,Month-to-month,Yes,Electronic check,104.80,3046.05,Yes,1,84,5003,Moved
4,0280-XJGEX,1,United States,California,Los Angeles,90015,"34.039224, -118.266293",34.039224,-118.266293,Male,No,No,Yes,49,Yes,Yes,Fiber optic,No,Yes,Yes,No,Yes,Yes,Month-to-month,Yes,Bank transfer (automatic),103.70,5036.3,Yes,1,89,5340,Competitor had better devices


In [6]:
df.shape

(7043, 33)

In [7]:
# Identify columns with only one unique value
for col in df.columns:
    if df[col].nunique() == 1:
        print(f"{col}: {df[col].unique()}")

Count: [1]
Country: ['United States']
State: ['California']


### 📊 Observation 1: Constant Columns

The following columns have only one unique value across all rows:

- Count  
- Country  
- State  

These columns do not provide any useful information for modeling and can be removed.

In [8]:
df['Zip Code'].nunique()

1652

In [9]:
df.groupby('City')['Zip Code'].nunique()

City
Acampo          1
Acton           1
Adelanto        1
Adin            1
Agoura Hills    1
               ..
Yreka           1
Yuba City       2
Yucaipa         1
Yucca Valley    1
Zenia           1
Name: Zip Code, Length: 1129, dtype: int64

### 📊 Observation 2: Location Features

- Most cities are associated with only one zip code  
- Some large cities (e.g., Los Angeles, San Diego) have multiple zip codes  

This indicates that:
- Zip Code may contain more granular information than City  
- However, both may introduce high cardinality  

These features should be handled carefully or excluded during modeling.

In [10]:
df.dtypes

CustomerID            object
Count                  int64
Country               object
State                 object
City                  object
Zip Code               int64
Lat Long              object
Latitude             float64
Longitude            float64
Gender                object
Senior Citizen        object
Partner               object
Dependents            object
Tenure Months          int64
Phone Service         object
Multiple Lines        object
Internet Service      object
Online Security       object
Online Backup         object
Device Protection     object
Tech Support          object
Streaming TV          object
Streaming Movies      object
Contract              object
Paperless Billing     object
Payment Method        object
Monthly Charges      float64
Total Charges         object
Churn Label           object
Churn Value            int64
Churn Score            int64
CLTV                   int64
Churn Reason          object
dtype: object

### 📊 Observation 3: Data Types

- Dataset contains both categorical and numerical features  
- Some numerical-looking columns may require type conversion  
- Target variable: **Churn Label / Churn Value**

### 🚨 Key Findings from Data Understanding

- Some columns are irrelevant (CustomerID, Count, Country, State)  
- Location-based features have high cardinality  
- Dataset contains both categorical and numerical features  
- Target variable is available in both label and numeric format  

Next step: Data Cleaning and Preprocessing

---

## 🧹 Step 4: Data Cleaning & Feature Selection

In this step, we clean the dataset by:

- Removing irrelevant and redundant features  
- Handling data type inconsistencies  
- Preparing the dataset for analysis and modeling  

The goal is to retain only meaningful features that contribute to predicting churn.

### 📊 Feature Removal Justification

The following columns were removed:

- **CustomerID** → Unique identifier, no predictive value  
- **Count** → Constant column (no variation)  
- **Country, State** → Single value across dataset  
- **Lat Long** → Redundant (Latitude & Longitude already exist)

Removing these helps reduce noise and improves model efficiency.

In [12]:
# Drop irrelevant and redundant columns
df = df.drop(columns=[
    'CustomerID',
    'Count',
    'Country',
    'State',
    'Lat Long'
])

### 🎯 Target Variable Selection

- **Churn Value (0 / 1)** is used as the target variable  
- Other churn-related columns are removed to avoid data leakage  

Final Target: **Churn**

In [13]:
# Use numeric target variable
df['Churn'] = df['Churn Value']

# Drop redundant churn columns
df = df.drop(columns=['Churn Label', 'Churn Value', 'Churn Score', 'Churn Reason'])

### 📊 Data Type Correction

- `Total Charges` was stored as an object type  
- Converted to numeric format for proper analysis  

Missing values introduced during conversion will be handled next.

In [14]:
# Convert Total Charges to numeric
df['Total Charges'] = pd.to_numeric(df['Total Charges'], errors='coerce')

In [15]:
# Check missing values
df.isnull().sum()

City                  0
Zip Code              0
Latitude              0
Longitude             0
Gender                0
Senior Citizen        0
Partner               0
Dependents            0
Tenure Months         0
Phone Service         0
Multiple Lines        0
Internet Service      0
Online Security       0
Online Backup         0
Device Protection     0
Tech Support          0
Streaming TV          0
Streaming Movies      0
Contract              0
Paperless Billing     0
Payment Method        0
Monthly Charges       0
Total Charges        11
CLTV                  0
Churn                 0
dtype: int64

In [16]:
# Fill missing values in Total Charges with median
df['Total Charges'].fillna(df['Total Charges'].median(), inplace=True)

C:\Users\NIHAR\AppData\Local\Temp\ipykernel_15368\2933346659.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['Total Charges'].fillna(df['Total Charges'].median(), inplace=True)


### 📊 Missing Value Handling

- Missing values were found in **Total Charges**  
- These were replaced using the **median value** to avoid skewing the distribution  

This ensures the dataset remains consistent for modeling.

In [18]:
# Drop potential data leakage columns
df = df.drop(columns=['CLTV'])

### ⚠️ Data Leakage Prevention

- **CLTV (Customer Lifetime Value)** may include future information  
- Keeping it can artificially inflate model performance  

Therefore, it is removed to ensure realistic predictions.

In [19]:
df.shape

(7043, 24)

In [20]:
df.head()

,City,Zip Code,Latitude,Longitude,Gender,Senior Citizen,Partner,Dependents,Tenure Months,Phone Service,Multiple Lines,Internet Service,Online Security,Online Backup,Device Protection,Tech Support,Streaming TV,Streaming Movies,Contract,Paperless Billing,Payment Method,Monthly Charges,Total Charges,Churn
0,Los Angeles,90003,33.964131,-118.272783,Male,No,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,1
1,Los Angeles,90005,34.059281,-118.307420,Female,No,No,Yes,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,1
2,Los Angeles,90006,34.048013,-118.293953,Female,No,No,Yes,8,Yes,Yes,Fiber optic,No,No,Yes,No,Yes,Yes,Month-to-month,Yes,Electronic check,99.65,820.50,1
3,Los Angeles,90010,34.062125,-118.315709,Female,No,Yes,Yes,28,Yes,Yes,Fiber optic,No,No,Yes,Yes,Yes,Yes,Month-to-month,Yes,Electronic check,104.80,3046.05,1
4,Los Angeles,90015,34.039224,-118.266293,Male,No,No,Yes,49,Yes,Yes,Fiber optic,No,Yes,Yes,No,Yes,Yes,Month-to-month,Yes,Bank transfer (automatic),103.70,5036.30,1


### ✅ Final Dataset Status

- Dataset is now cleaned and ready for analysis  
- Irrelevant and redundant features removed  
- Target variable properly defined  
- Missing values handled  

Next step: Exploratory Data Analysis (EDA)

---

## 📊 Step 5: Exploratory Data Analysis (EDA)

In this section, we analyze customer behavior to identify patterns associated with churn.

The goal is to:
- Understand key drivers of churn  
- Identify high-risk customer segments  
- Extract actionable business insights  

In [21]:
df['Churn'].value_counts(normalize=True)

Churn
0    0.73463
1    0.26537
Name: proportion, dtype: float64

### 📊 Churn Distribution

- The dataset is imbalanced  
- Majority of customers do not churn  
- Minority churn class needs special attention during modeling  

⚠️ This justifies the use of metrics like **F1-score, ROC-AUC instead of accuracy**

In [22]:
df.groupby('Tenure Months')['Churn'].mean()

Tenure Months
0     0.000000
1     0.619902
2     0.516807
3     0.470000
4     0.471591
        ...   
68    0.090000
69    0.084211
70    0.092437
71    0.035294
72    0.016575
Name: Churn, Length: 73, dtype: float64

### 📊 Tenure vs Churn

- Customers with low tenure have significantly higher churn rates  
- Long-term customers are more stable  

💡 Insight:
- Early-stage customers are high-risk  
- Retention strategies should focus on **first few months**

In [23]:
df.groupby('Contract')['Churn'].mean().sort_values()

Contract
Two year          0.028319
One year          0.112695
Month-to-month    0.427097
Name: Churn, dtype: float64

### 📊 Contract Type vs Churn

- Month-to-month customers show highest churn  
- Long-term contracts (1 year / 2 year) reduce churn  

💡 Insight:
- Lock-in contracts improve retention  
- Offering discounts for long-term plans can reduce churn

In [24]:
df.groupby('Payment Method')['Churn'].mean().sort_values()

Payment Method
Credit card (automatic)      0.152431
Bank transfer (automatic)    0.167098
Mailed check                 0.191067
Electronic check             0.452854
Name: Churn, dtype: float64

### 📊 Payment Method vs Churn

- Electronic check users have higher churn  
- Automatic payment methods show lower churn  

💡 Insight:
- Customers using manual payment methods are less committed  
- Incentivizing auto-pay could reduce churn

In [26]:
df.groupby('Churn')['Monthly Charges'].mean()

Churn
0    61.265124
1    74.441332
Name: Monthly Charges, dtype: float64

### 📊 Monthly Charges vs Churn

- Customers with higher monthly charges tend to churn more  

💡 Insight:
- High pricing may be a churn driver  
- Price optimization or bundled offers may help retention

In [28]:
df.groupby('Internet Service')['Churn'].mean()

Internet Service
DSL            0.189591
Fiber optic    0.418928
No             0.074050
Name: Churn, dtype: float64

### 📊 Internet Service vs Churn

- Fiber optic users show higher churn compared to DSL  

💡 Insight:
- Premium services may come with higher expectations  
- Poor experience → higher churn

In [30]:
services = [
    'Online Security', 'Online Backup', 'Device Protection',
    'Tech Support', 'Streaming TV', 'Streaming Movies'
]

for col in services:
    print(col)
    print(df.groupby(col)['Churn'].mean())
    print()

Online Security
Online Security
No                     0.417667
No internet service    0.074050
Yes                    0.146112
Name: Churn, dtype: float64

Online Backup
Online Backup
No                     0.399288
No internet service    0.074050
Yes                    0.215315
Name: Churn, dtype: float64

Device Protection
Device Protection
No                     0.391276
No internet service    0.074050
Yes                    0.225021
Name: Churn, dtype: float64

Tech Support
Tech Support
No                     0.416355
No internet service    0.074050
Yes                    0.151663
Name: Churn, dtype: float64

Streaming TV
Streaming TV
No                     0.335231
No internet service    0.074050
Yes                    0.300702
Name: Churn, dtype: float64

Streaming Movies
Streaming Movies
No                     0.336804
No internet service    0.074050
Yes                    0.299414
Name: Churn, dtype: float64



### 📊 Service Features vs Churn

- Customers without additional services (security, backup, support) churn more  
- Customers using multiple services are more loyal  

💡 Insight:
- Higher service engagement reduces churn  
- Bundling services can improve retention

In [31]:
df.groupby('Senior Citizen')['Churn'].mean()

Senior Citizen
No     0.236062
Yes    0.416813
Name: Churn, dtype: float64

### 📊 Demographics vs Churn

- Senior citizens may have slightly higher churn  

💡 Insight:
- Different customer segments require different retention strategies

## 🚨 Key Insights from EDA

- Customers with **low tenure** are most likely to churn  
- **Month-to-month contracts** have highest churn rates  
- **High monthly charges** increase churn probability  
- **Electronic check users** show higher churn  
- Customers without **additional services** are more likely to leave  

---

## 💡 Business Takeaways

- Focus retention efforts on **new customers**  
- Promote **long-term contracts**  
- Encourage **auto-pay methods**  
- Bundle services to increase engagement  